In [1]:
# %env REGIONS=HIM

In [2]:
# Cell 1: Imports & Setup
import pandas as pd
from pathlib import Path
import json, gc
import numpy as np
import polars as pl
from sklearn.feature_selection import mutual_info_classif
import spatial_split as ss
from config import regions as config_regions

output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

import os
# Regions to process: defaults to the regions in config.py, or set via the REGIONS environment variable (comma separated)
TARGET_REGIONS = os.environ.get('REGIONS', ','.join(config_regions)).split(',')

# Predictor exclusions: metadata, target, and null-heavy composite bands (lat/lon are kept as predictors)
cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year', 'mosaic_version', 'split',
    'BareEarth', 'StableWater', 'TransientWater',   # ancillary mask columns, not predictors
]
pixel_quality_condition = ss.QUALITY          # usable_count >= 5 & quarters_present >= 3
class_remap_expr = ss._remap()                # Level 2 -> Level 1 class remap, same as every other step

In [3]:
# Cell 2: Drop redundant features, using train-split pixels of the usable years
# Redundancy: |Spearman rho| between features. Relevance: mutual information with the class (class codes are used as labels,
# never as numbers). Features are visited from most to least relevant; a feature is kept unless it has |rho| > THRESHOLD
# with a feature already kept. So no kept pair exceeds THRESHOLD, and a feature is only dropped because of a kept one.
THRESHOLD = 0.80
CORR_SAMPLE = 50_000   # pixels for the correlation matrix
MI_BALANCED = True     # mutual information on a class-balanced draw, so rare classes count as much as common ones (macro-F1)
MI_PER_CLASS = 5_000   # pixels per class for that draw (all pixels of classes that have fewer)
MI_SAMPLE = 50_000     # pixels for mutual information when MI_BALANCED is False


def prune_redundant(X_corr, X_mi, y_mi, feature_cols, threshold):
    """Greedy redundancy pruning.
    Returns: kept features (most relevant first), {dropped feature: kept feature it correlates with},
    zero-variance features, relevance scores."""
    n_nan = np.isnan(X_corr).sum(0) + np.isnan(X_mi).sum(0)
    if n_nan.any():
        bad = [c for c, n in zip(feature_cols, n_nan) if n]
        raise ValueError(f"Missing values in {bad}: exclude these columns or handle the nulls first.")

    std = X_corr.std(axis=0)
    zero_var = [c for c, s in zip(feature_cols, std) if s == 0]
    use = [i for i, s in enumerate(std) if s > 0]
    cols = [feature_cols[i] for i in use]

    # Relevance to the class (all features are continuous measurements, hence discrete_features=False)
    relevance = pd.Series(
        mutual_info_classif(X_mi[:, use], y_mi, discrete_features=False, random_state=42),
        index=cols,
    )

    # Redundancy: Spearman = Pearson correlation of the ranks
    ranks = pd.DataFrame(X_corr[:, use]).rank(method='average').to_numpy()
    corr = pd.DataFrame(np.abs(np.corrcoef(ranks, rowvar=False)), index=cols, columns=cols)

    kept, absorbed_by = [], {}
    for feat in relevance.sort_values(ascending=False, kind='stable').index:
        partners = corr.loc[feat, kept]
        if kept and partners.max() > threshold:
            absorbed_by[feat] = partners.idxmax()   # the kept feature that stands in for it
        else:
            kept.append(feat)
    return kept, absorbed_by, zero_var, relevance


for target_region in TARGET_REGIONS:
    print(f"\n{'='*50}\nProcessing region: {target_region}\n{'='*50}")
    years = ss.study_years(target_region)
    assert years, f'run 03_split_and_years.ipynb for {target_region} first'

    lazy_ds = ss.scan_years(target_region, years)
    feature_cols = [c for c in lazy_ds.collect_schema().names() if c not in cols_to_drop]

    df_collected = (
        ss.add_split(lazy_ds.with_columns(class_remap_expr), target_region)
        .filter(pixel_quality_condition & (pl.col('split') == 'train'))
        .select(feature_cols + ['class'])
        .collect()
    )
    print(f"[{target_region}] Train-split gated rows: {df_collected.height:,}")

    df_corr = df_collected.sample(n=min(CORR_SAMPLE, df_collected.height), seed=42)
    if MI_BALANCED:
        cls = df_collected['class'].to_numpy()
        rng = np.random.default_rng(42)
        idx = np.concatenate([rng.permutation(np.flatnonzero(cls == c))[:MI_PER_CLASS] for c in np.unique(cls)])
        df_mi = (df_collected.with_row_index('_row')
                 .filter(pl.col('_row').is_in(pl.Series(idx)))
                 .drop('_row'))
    else:
        df_mi = df_collected.sample(n=min(MI_SAMPLE, df_collected.height), seed=42)
    del df_collected; gc.collect()

    X_corr = df_corr.select(feature_cols).to_numpy().astype(np.float64)
    X_mi = df_mi.select(feature_cols).to_numpy().astype(np.float64)
    y_mi = df_mi['class'].to_numpy().astype(np.int32)
    del df_corr, df_mi; gc.collect()
    mi_desc = f"class-balanced, up to {MI_PER_CLASS:,} per class" if MI_BALANCED else "random"
    print(f"[{target_region}] Correlation sample: {len(X_corr):,} rows; mutual-information sample: {len(y_mi):,} rows ({mi_desc})")

    kept, absorbed_by, zero_var, relevance = prune_redundant(X_corr, X_mi, y_mi, feature_cols, THRESHOLD)
    if zero_var:
        print(f"[{target_region}] Dropped {len(zero_var)} zero-variance feature(s): {zero_var}")
    dropped = sorted(absorbed_by) + zero_var

    out = output_dir / f"{target_region}_uncorrelated_features.json"
    json.dump({
        "region": target_region,
        "years": years,
        "corr_threshold": THRESHOLD,
        "corr_method": "spearman",
        "relevance_method": "mutual_info_classif",
        "mi_class_balanced": MI_BALANCED,
        "mi_per_class": MI_PER_CLASS if MI_BALANCED else None,
        "corr_sample_rows": int(len(X_corr)),
        "mi_sample_rows": int(len(y_mi)),
        "total_raw_features": len(feature_cols),
        "uncorrelated_features": kept,          # ordered by relevance, most relevant first
        "dropped_features": dropped,
        "absorbed_by": absorbed_by,             # dropped feature -> kept feature it correlates with above the threshold
        "zero_variance_features": zero_var,
        "relevance_scores": {k: round(float(v), 6) for k, v in relevance.items()},
    }, open(out, "w"), indent=2)
    print(f"[{target_region}] raw {len(feature_cols)} -> dropped {len(dropped)} -> kept {len(kept)}  ({out})")
    print(f"[{target_region}] Kept: {kept}")
    del X_corr, X_mi, y_mi; gc.collect()


Processing region: HIM
[HIM] Train-split gated rows: 955,449


/tmp/ipykernel_60070/1888767516.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('_row').is_in(pl.Series(idx)))


[HIM] Correlation sample: 50,000 rows; mutual-information sample: 55,000 rows (class-balanced, up to 5,000 per class)
[HIM] raw 103 -> dropped 61 -> kept 42  (data/feature_selection_artifacts/HIM_uncorrelated_features.json)
[HIM] Kept: ['lon', 'lat', 'elevation', 'ndvi_median', 'tcw_median', 'tcb_median', 'aspect_cos', 'aspect_sin', 'green_median_wet', 'ndfi_mad', 'ndmi_median', 'tcg_median_wet', 'slope', 'evi2_swing', 'blue_median_dry', 'mndwi_median_dry', 'tcg_median_dry', 'ndvi_swing', 'shade_median', 'soil_swing', 'ndmi_swing', 'nir_swing', 'ndvi_mad', 'swir2_swing', 'ndmi_mad', 'nir_mad', 'red_swing', 'gv_mad', 'shade_swing', 'npv_median', 'tcb_swing', 'red_mad', 'tcb_mad', 'mndwi_mad', 'tcw_mad', 'bci_mad', 'soil_mad', 'mndwi_swing', 'green_mad', 'blue_mad', 'npv_mad', 'npv_swing']

Processing region: IGP
[IGP] Train-split gated rows: 1,693,033


/tmp/ipykernel_60070/1888767516.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('_row').is_in(pl.Series(idx)))


[IGP] Correlation sample: 50,000 rows; mutual-information sample: 50,000 rows (class-balanced, up to 5,000 per class)
[IGP] raw 103 -> dropped 58 -> kept 45  (data/feature_selection_artifacts/IGP_uncorrelated_features.json)
[IGP] Kept: ['lon', 'lat', 'elevation', 'ndvi_p25', 'mndwi_median', 'swir2_median', 'swir1_median_dry', 'ndmi_median_dry', 'ndfi_swing', 'red_median_wet', 'aspect_cos', 'ndfi_mad', 'aspect_sin', 'shade_median', 'slope', 'soil_swing', 'ndmi_median_wet', 'ndmi_swing', 'gv_swing', 'hand', 'green_median_dry', 'npv_median', 'swir1_swing', 'ndmi_mad', 'ndvi_swing', 'swir2_mad', 'mndwi_mad', 'npv_median_wet', 'red_swing', 'npv_median_dry', 'mndwi_swing', 'soil_mad', 'npv_swing', 'shade_swing', 'nir_swing', 'swir1_mad', 'red_mad', 'bci_mad', 'nir_mad', 'shade_mad', 'blue_mad', 'tcb_swing', 'npv_mad', 'green_mad', 'tcb_mad']

Processing region: DES
[DES] Train-split gated rows: 2,742,402


/tmp/ipykernel_60070/1888767516.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('_row').is_in(pl.Series(idx)))


... (5 outputs left out)

/tmp/ipykernel_60070/1888767516.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('_row').is_in(pl.Series(idx)))


[WGR] Correlation sample: 50,000 rows; mutual-information sample: 53,405 rows (class-balanced, up to 5,000 per class)
[WGR] raw 103 -> dropped 65 -> kept 38  (data/feature_selection_artifacts/WGR_uncorrelated_features.json)
[WGR] Kept: ['lat', 'elevation', 'ndvi_median_dry', 'swir1_median', 'mndwi_median', 'aspect_cos', 'aspect_sin', 'red_median', 'evi2_median_wet', 'ndmi_median_dry', 'shade_median_dry', 'swir2_swing', 'hand', 'slope', 'ndmi_swing', 'gv_swing', 'blue_median_wet', 'ndmi_median_wet', 'ndvi_swing', 'npv_median', 'swir2_mad', 'mndwi_swing', 'ndmi_mad', 'mndwi_mad', 'red_mad', 'npv_median_wet', 'blue_swing', 'npv_swing', 'bci_mad', 'nir_swing', 'shade_swing', 'blue_mad', 'tcb_swing', 'nir_mad', 'shade_mad', 'tcb_mad', 'npv_mad', 'green_mad']

Processing region: ICR
[ICR] Train-split gated rows: 1,439,109


/tmp/ipykernel_60070/1888767516.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col('_row').is_in(pl.Series(idx)))


[ICR] Correlation sample: 50,000 rows; mutual-information sample: 55,000 rows (class-balanced, up to 5,000 per class)
[ICR] raw 103 -> dropped 59 -> kept 44  (data/feature_selection_artifacts/ICR_uncorrelated_features.json)
[ICR] Kept: ['lat', 'lon', 'elevation', 'ndvi_p25', 'mndwi_median', 'swir2_median', 'aspect_cos', 'aspect_sin', 'shade_median', 'ndfi_swing', 'red_median_wet', 'slope', 'soil_swing', 'ndmi_median_wet', 'ndmi_swing', 'tcb_median_dry', 'gv_swing', 'red_median_dry', 'ndvi_swing', 'hand', 'tcw_swing', 'red_swing', 'tcb_median_wet', 'npv_median', 'ibi_mad', 'tcg_mad', 'mndwi_swing', 'mndwi_mad', 'swir2_mad', 'npv_median_dry', 'soil_mad', 'npv_median_wet', 'bci_mad', 'red_mad', 'npv_swing', 'blue_mad', 'nir_swing', 'tcb_swing', 'shade_swing', 'green_mad', 'tcb_mad', 'shade_mad', 'nir_mad', 'npv_mad']
